In [2]:
import torch
import torch.nn as nn

In [3]:
NUM_FRAMES=16
IMAGE_SIZE=112
D_MODEL=256
NUM_HEADS=4
NUM_LAYERS=2
NUM_CLASSES=2
DROPOUT=0.3
DEVICE=torch.device("cuda" if torch.cuda.is_available()else "cpu")
print("device: ",DEVICE)

device:  cpu


In [8]:
class CNN3D(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2)),

            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2)),

            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 2, 2))
        )

    def forward(self, x):
        x = self.features(x)
        x = x.mean(dim=(-1, -2))
        return x.transpose(1, 2)

In [9]:
class HybridFightDetector(nn.Module):
    def __init__(
        self,
        num_frames=NUM_FRAMES,
        d_model=D_MODEL,
        num_heads=NUM_HEADS,
        num_layers=NUM_LAYERS,
        num_classes=NUM_CLASSES,
        dropout=DROPOUT
    ):
        super().__init__()

        self.cnn = CNN3D()
        self.projection = nn.Linear(128, d_model)

        self.position_embedding = nn.Parameter(
            torch.randn(1, num_frames, d_model) * 0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.classifier = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

    def forward(self, x):
        x = self.cnn(x)
        x = self.projection(x)
        x = x + self.position_embedding[:, :x.size(1), :]
        x = self.transformer(x)
        x = x.mean(dim=1)
        return self.classifier(x)

In [10]:
model = HybridFightDetector().to(DEVICE)

print(model)

HybridFightDetector(
  (cnn): CNN3D(
    (features): Sequential(
      (0): Conv3d(3, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (1): BatchNorm3d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, dilation=1, ceil_mode=False)
      (4): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (5): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (6): ReLU(inplace=True)
      (7): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, dilation=1, ceil_mode=False)
      (8): Conv3d(64, 128, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
      (9): BatchNorm3d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (10): ReLU(inplace=True)
      (11): MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2), padding=0, d

C:\Users\Yousaf Bin Aamir\AppData\Local\Temp\ipykernel_18808\3737342175.py:29: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


In [11]:
sample_video = torch.randn(
    2, 3, NUM_FRAMES, IMAGE_SIZE, IMAGE_SIZE
).to(DEVICE)

model.eval()

with torch.no_grad():
    output = model(sample_video)

print("Input shape:", sample_video.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 3, 16, 112, 112])
Output shape: torch.Size([2, 2])


In [12]:
total_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Trainable parameters: {total_params:,}")

Trainable parameters: 1,897,410
